In [13]:
import argparse, csv, glob, os, re, statistics as st
from collections import defaultdict

In [25]:
ap = argparse.ArgumentParser()

ap.add_argument("root", nargs="?", default=".")
ap.add_argument("--secs", type=float, default=180)
ap.add_argument("--warm", type=float, default=30)
ap.add_argument("--out", default="summary_ro.csv")

a, unknown = ap.parse_known_args()

ROOT = a.root
SECS = a.secs
WARM = a.warm
out = os.path.join(ROOT, OUT)

SYS = {("k10", "depart"): "DePart, Alg. 3 (k=10, fixed)",
       ("k10", "index"): "Random + index (k=10)",
       ("k1", "obladi"): "Obladi-style (k=1)",
       ("kcap", "depart"): "DePart, Alg. 4 (K on demand)",
       ("kcap", "index"): "Random + index (same K)"}
METRICS = ["failed", "tps", "commit", "heavy", "lat", "e2e_p50", "e2e_p95", "dropped", "slots", "real", "shards", "K"]

In [29]:
def one_run(prefix):
    ends, t = {}, 0.0
    for r in csv.DictReader(open(prefix + "_epochs.csv")):
        t += float(r["epoch_ms"]) / 1000; ends[int(r["epoch"])] = t
    lo, hi = a.warm, a.secs
    first, done, lat = {}, {}, []
    for r in csv.DictReader(open(prefix + "_attempts.csv")):
        tid, end = r["client_txn_id"], ends.get(int(r["epoch"]))
        if end is None:
            continue
        if r["attempt"] == "1":
            first[tid] = (end, r["txn_type"])
        if r["committed"] == "1":
            done[tid] = end
            if lo <= end <= hi:
                lat.append(float(r["latency_ms"]) / 1000)
    win = [tid for tid, (e, _) in first.items() if lo <= e <= hi]
    heavy = [tid for tid in win if first[tid][1] in ("T4", "T5")]
    types = [r for r in csv.DictReader(open(prefix + "_types.csv")) if int(r["committed"]) > 0]
    n = sum(int(r["committed"]) for r in types)
    wavg = lambda c: sum(int(r["committed"]) * float(r[c]) for r in types) / n / 1000
    pl = next(csv.DictReader(open(prefix + "_placement.csv")))
    com = int(pl["committed"])
    return dict(
        tps=sum(lo <= e <= hi for e in done.values()) / (hi - lo),
        commit=sum(tid in done for tid in win) / len(win) if win else float("nan"),
        heavy=sum(tid in done for tid in heavy) / len(heavy) if heavy else float("nan"),
        lat=st.median(lat) if lat else float("nan"),
        e2e_p50=wavg("e2e_p50_ms"), e2e_p95=wavg("e2e_p95_ms"),
        dropped=float(types[0]["dropped"]) if types else 0.0,
        slots=float(pl["slots_per_committed_txn"]),
        real=(int(pl["real_read_slots"]) + int(pl["real_write_slots"])) / com if com else float("nan"),
        shards=float(pl["avg_shards_per_read_round"]), K=float(pl["k"]), failed=float("nan"))

runs = defaultdict(dict)    # (system, rate) -> {seed: metrics}
pat = re.compile(r"tpcc_W5_(k10|k1|kcap)_(depart|index|obladi)_r(\d+)_types\.csv$")
for f in sorted(glob.glob(os.path.join(ROOT, "**", "oasis_W5_*_summary_R*.csv"), recursive=True)):
    m = pat.search(os.path.basename(f))
    if not m or (m[1], m[2]) not in SYS:
        continue
    seed = next((p.replace("oasis_", "") for p in reversed(f.split(os.sep))
                 if re.fullmatch(r"(oasis_)?seed\d+", p)), "seed?")
    prefix = f[:-len("_types.csv")]
    if not all(os.path.exists(prefix + s) for s in ("_epochs.csv", "_attempts.csv", "_placement.csv")):
        print("incomplete run, skipped:", prefix); continue
    runs[(SYS[(m[1], m[2])], int(m[3]))][seed] = one_run(prefix)

In [30]:
def oasis_run(summ, types):
    s = next(csv.DictReader(open(summ)))
    ty = list(csv.DictReader(open(types)))
    done = sum(int(r["completed"]) for r in ty)
    wavg = lambda c: sum(int(r["completed"]) * float(r[c]) for r in ty) / done if done else float("nan")
    hv = [r for r in ty if r["txn_type"] in ("T4", "T5")]
    nan = float("nan")
    return dict(tps=float(s["throughput_txn_s"]), commit=int(s["completed"]) / int(s["arrived"]),
                heavy=sum(int(r["completed"]) for r in hv) / max(1, sum(int(r["arrived"]) for r in hv)),
                lat=nan, e2e_p50=wavg("lat_p50_s"), e2e_p95=wavg("lat_p95_s"),
                dropped=float(s["dropped"]), failed=int(s["failed"]) / int(s["arrived"]),
                slots=nan, real=nan, shards=nan, K=5.0)

opat = re.compile(r"oasis_W5_k(\d+)_r(\d+)_summary_R(\d+)\.csv$")
for f in sorted(glob.glob(os.path.join(a.root, "**", "oasis_W5_*_summary_R*.csv"), recursive=True)):
    m = opat.search(os.path.basename(f))
    t = f.replace("_summary_", "_types_")
    if not m or not os.path.exists(t):
        continue
    seed = next((p.replace("oasis_", "") for p in reversed(f.split(os.sep))
                 if re.fullmatch(r"(oasis_)?seed\d+", p)), "seed?")
    name = f"OasisDB (R={m[3]}, {m[1]} exec.)"
    if name not in SYS.values():
        SYS[("oasis", m[3])] = name
    runs[(name, int(m[2]))][seed] = oasis_run(f, t)

order = list(SYS.values())
rows = []
for (s, rate) in sorted(runs, key=lambda k: (order.index(k[0]), k[1])):
    v = runs[(s, rate)]
    row = {"system": s, "offered_txn_s": rate, "n_runs": len(v), "seeds": " ".join(sorted(v))}
    for m in METRICS:
        x = [r[m] for r in v.values() if r[m] == r[m]]
        row[f"{m}_mean"] = f"{st.mean(x):.6g}" if x else ""
        row[f"{m}_std"] = f"{st.stdev(x):.4g}" if len(x) > 1 else ("0" if x else "")
    rows.append(row)
    print(f"{s:32s} {rate:5d}  n={len(v)}  tps={row['tps_mean']:>8s}  e2e_p50={row['e2e_p50_mean']:>7s}s  "
          f"heavy={row['heavy_mean']:>6s}  slots={row['slots_mean']}")

out = os.path.join(ROOT, OUT)

with open(out, "w", newline="") as fh:
    w = csv.DictWriter(
        fh,
        fieldnames=["system", "offered_txn_s", "n_runs", "seeds"] +
                   [f"{m}_{s}" for m in METRICS for s in ("mean", "std")]
    )
    w.writeheader()
    w.writerows(rows)

print("wrote", out)

NotADirectoryError: [Errno 20] Not a directory: '/Users/xiningyuan/Library/Jupyter/runtime/kernel-08ac0ddd-bdbf-44ae-a0ee-4f3b5f019996.json/summary_ro.csv'